## Init

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA, Imputer, StringIndexer
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import col, when, sum, isnan, udf
from pyspark.sql.types import StringType

import matplotlib.pyplot as plt
import numpy as np

spark = SparkSession.builder \
    .appName("SpotifyRecommandation") \
    .config("spark.driver.memory", "16g") \
    .config("spark.executor.memory", "16g") \
    .config("zspark.executor.cores", "2") \
    .getOrCreate()

df_songs = spark.read.csv(
    "hdfs://hadoop-namenode-1:8020/data/data.csv",
    header=True, inferSchema=True
).dropDuplicates()

df_genres = spark.read.csv(
    "hdfs://hadoop-namenode-1:8020/data/data_by_genres.csv", 
    header=True, inferSchema=True
)

df_songs.printSchema()
df_songs.show(5)

root
 |-- valence: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- acousticness: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- danceability: string (nullable = true)
 |-- duration_ms: string (nullable = true)
 |-- energy: string (nullable = true)
 |-- explicit: string (nullable = true)
 |-- id: string (nullable = true)
 |-- instrumentalness: string (nullable = true)
 |-- key: string (nullable = true)
 |-- liveness: string (nullable = true)
 |-- loudness: string (nullable = true)
 |-- mode: string (nullable = true)
 |-- name: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- speechiness: string (nullable = true)
 |-- tempo: string (nullable = true)

+-------+----+------------+--------------------+------------+-----------+------+--------+--------------------+----------------+---+--------+--------+----+--------------------+----------+------------+-----------+-------+
|valence|year|a

In [2]:
audio_cols = [
    "danceability","energy","valence","acousticness",
    "instrumentalness","liveness","speechiness","tempo"
]

for c in audio_cols:
    df_songs = df_songs.withColumn(c, col(c).cast("double"))
    df_genres = df_genres.withColumn(c, col(c).cast("double"))

df_songs = df_songs.withColumn("popularity", col("popularity").cast("double"))
df_genres = df_genres.withColumn("popularity", col("popularity").cast("double"))


In [3]:
imputer_songs = Imputer(
    inputCols=audio_cols,
    outputCols=audio_cols
).setStrategy("mean")

df_songs_imputed = imputer_songs.fit(df_songs).transform(df_songs)
print("DataFrame des chansons nettoyé.")


imputer_genres = Imputer(
    inputCols=audio_cols,
    outputCols=audio_cols
).setStrategy("mean")

df_genres_imputed = imputer_genres.fit(df_genres).transform(df_genres)
print("DataFrame des genres nettoyé.")


print("\nSong data overview:")
df_songs_imputed.select(audio_cols).show(5)

print("\nOverview of gender data:")
df_genres_imputed.select(audio_cols).show(5)

print("\nRemaining zero values in songs :")
df_songs_imputed.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in audio_cols
]).groupBy().sum().show()

print("\nRemaining zero values in songs gender :")
df_genres_imputed.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in audio_cols
]).groupBy().sum().show()

DataFrame des chansons nettoyé.
DataFrame des genres nettoyé.

Song data overview:
+------------+------+-------+------------+----------------+--------+-----------+-------+
|danceability|energy|valence|acousticness|instrumentalness|liveness|speechiness|  tempo|
+------------+------+-------+------------+----------------+--------+-----------+-------+
|       0.573| 0.246|  0.965|       0.907|           0.867|  0.0752|      0.224|205.419|
|       0.424| 0.309|  0.327|       0.996|           0.946|   0.175|     0.0591| 127.24|
|       0.514| 0.158|   0.75|       0.995|           0.943|   0.273|     0.0488| 123.55|
|       0.818| 0.171|  0.934|       0.996|           0.919|   0.112|      0.295|111.451|
|       0.525| 0.121|  0.551|       0.996|           0.924|   0.101|     0.0744|121.913|
+------------+------+-------+------------+----------------+--------+-----------+-------+
only showing top 5 rows


Overview of gender data:
+-------------------+-------------------+-------------------+----

In [4]:
assembler_songs = VectorAssembler(
    inputCols=audio_cols, 
    outputCol="features", 
    handleInvalid="skip"
)
df_songs_vec = assembler_songs.transform(df_songs_imputed)

assembler_genres = VectorAssembler(
    inputCols=audio_cols, 
    outputCol="genre_features", 
    handleInvalid="skip"
)
df_genres_vec = assembler_genres.transform(df_genres_imputed)

In [5]:
genres_lookup = df_genres_vec.select("genres", "genre_features").collect()

def get_closest_genre(song_features):
    best_genre = None
    min_dist = float('inf')
    
    for row in genres_lookup:
        dist = np.linalg.norm(song_features.toArray() - row.genre_features.toArray())
        if dist < min_dist:
            min_dist = dist
            best_genre = row.genres
            
    return best_genre

closest_genre_udf = udf(get_closest_genre, StringType())

print("Assigning a genre to each song...")
df_songs_with_genre = df_songs_vec.withColumn("genre", closest_genre_udf(col("features")))

print("\Assignement complete! Preview:")
df_songs_with_genre.select("name", "artists", "genre").show(5, truncate=False)

Assigning a genre to each song...
\Assignement complete! Preview:
+-----------------------------------------+-----------------------------------+----------------------+
|name                                     |artists                            |genre                 |
+-----------------------------------------+-----------------------------------+----------------------+
|Weary Blues                              |['Louis Armstrong & His Hot Seven']|danseband             |
|Sobin Blue - Remasterizado               |['Francisco Canaro']               |danish electropop     |
|Ojerosa - Remasterizado                  |['Ignacio Corsini']                |spanish baroque       |
|No Folling - Instrumental (Remasterizado)|['Francisco Canaro']               |christmas instrumental|
|En la Cortada - Remasterizado            |['Ignacio Corsini']                |honky-tonk piano      |
+-----------------------------------------+-----------------------------------+----------------------+
only sh

In [6]:

string_indexer = StringIndexer(inputCol="genre", outputCol="genre_index", handleInvalid="keep")
model_indexer = string_indexer.fit(df_songs_with_genre)
df_final_features = model_indexer.transform(df_songs_with_genre)

print("The ‘gender’ column has been converted to a numeric index.")

final_assembler = VectorAssembler(
    inputCols=["features", "year", "genre_index"],
    outputCol="unscaled_final_features"
)
df_unscaled = final_assembler.transform(df_final_features)

final_scaler = StandardScaler(
    inputCol="unscaled_final_features",
    outputCol="final_features", 
    withStd=True,
    withMean=True
)
scaler_model_final = final_scaler.fit(df_unscaled)
df_ready_for_clustering = scaler_model_final.transform(df_unscaled)

print("The final vector has been successfully normalized.")

print("\nDataframe preview :")
df_ready_for_clustering.select("name", "genre", "year", "final_features").show(2, truncate=False)

The ‘gender’ column has been converted to a numeric index.
The final vector has been successfully normalized.

Dataframe preview :
+--------------------------+-----------------+----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|name                      |genre            |year|final_features                                                                                                                                                                                              |
+--------------------------+-----------------+----+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Weary Blues               |danseband        |1923|[0.2012945807312575,-0.03948991287920533,1.6588

# Normalize features

In [7]:
pca = PCA(k=10, inputCol="final_features", outputCol="pca_features")
model_pca = pca.fit(df_ready_for_clustering)
df_pca = model_pca.transform(df_ready_for_clustering)

explained = model_pca.explainedVariance.toArray()
cumvar    = np.cumsum(explained)

for i, (e, c) in enumerate(zip(explained, cumvar), start=1):
    print(f"PC{i:>2} : {e*100:6.2f} %    (cumul : {c*100:6.2f} %)")

PC 1 :  19.66 %    (cumul :  19.66 %)
PC 2 :  13.14 %    (cumul :  32.80 %)
PC 3 :  12.45 %    (cumul :  45.25 %)
PC 4 :  10.06 %    (cumul :  55.31 %)
PC 5 :  10.00 %    (cumul :  65.31 %)
PC 6 :   9.78 %    (cumul :  75.09 %)
PC 7 :   9.47 %    (cumul :  84.56 %)
PC 8 :   7.77 %    (cumul :  92.33 %)
PC 9 :   4.56 %    (cumul :  96.88 %)
PC10 :   3.12 %    (cumul : 100.00 %)


In [ ]:
K_PCA_OPTIMAL = 8

pca_final = PCA(k=K_PCA_OPTIMAL, inputCol="final_features", outputCol="pca_features")
model_pca_final = pca_final.fit(df_ready_for_clustering)
df_pca_final = model_pca_final.transform(df_ready_for_clustering)


In [9]:
df_pca_final.select("pca_features").show(5, truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|pca_features                                                                                                                                                     |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[0.8578432683314765,-2.5285527953969753,-0.9603476985411191,-0.4959866290159406,-0.015784039361871977,-1.073928371659948,0.9581735650427378,0.08705682485671509] |
|[2.229395395361416,-0.6659230255371261,-0.5991497660606764,0.3054862932038904,-0.022582787586647522,0.5285198479147714,0.7006765612651532,-0.0481584016954228]   |
|[1.3529595424399856,-1.8997781860728025,-0.5676459917664175,0.21012217303928976,-0.033752518437773216,0.3087234792309916,0.6037402010736839,0.009375781495797765]|
|[0.202372282340

# Clustering

In [12]:
df_to_export = df_pca_final.select("name", "artists", "popularity", "year", "genre", "pca_features")

pandas_df = df_to_export.toPandas()

pandas_df['pca_features'] = pandas_df['pca_features'].apply(lambda vec: list(vec.toArray()))

pandas_df.to_parquet("data_for_clustering.parquet")

print("Export success")

Export success


In [ ]:
print("Searching for optimal k with the Silhouette method...")
ks = range(2, 21) 
sil_scores = []

for k in ks:
    km = KMeans(featuresCol="pca_features", k=k, seed=42)
    preds = km.fit(df_pca_final_checkpointed).transform(df_pca_final_checkpointed)
    
    evaluator = ClusteringEvaluator(featuresCol="pca_features", metricName="silhouette")
    score = evaluator.evaluate(preds)
    
    sil_scores.append(score)
    print(f"K={k} → Score de Silhouette={score:.4f}")

plt.plot(ks, sil_scores, marker='x')
plt.xlabel('K')
plt.ylabel('Score de Silhouette')
plt.title('Score de Silhouette (sur données enrichies)')
plt.show()

In [ ]:
evaluator = ClusteringEvaluator(
    featuresCol="pca_features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

ks = list(range(2, 50)) 

sil_scores = []
for k in ks:
    km = KMeans(featuresCol="pca_features", k=k, seed=42)
    preds = km.fit(df_pca).transform(df_pca)
    score = evaluator.evaluate(preds)
    sil_scores.append(score)
    # print(f"K={k} → Silhouette={score:.4f}")

plt.plot(ks, sil_scores, marker='x')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.title('Silhouette (full dataset)')
plt.show()

In [ ]:
K_OPTIMAL = 5

print(f"Entraînement du modèle final avec k={K_OPTIMAL}...")

kmeans_final = KMeans(featuresCol="pca_features", k=K_OPTIMAL, seed=42)
modele_kmeans_final = kmeans_final.fit(df_pca)

df_clustered = modele_kmeans_final.transform(df_pca)

print("Clusters assignés avec succès !")
df_clustered.select("artists", "name", "prediction").show(10, truncate=False)

In [ ]:
pandas_df = df_clustered.select("name", "artists", "popularity", "prediction", "pca_features").toPandas()

pandas_df['pca_features'] = pandas_df['pca_features'].apply(lambda x: list(x.toArray()))

pandas_df.to_parquet("spotify_data_features.parquet")

print("Le fichier 'spotify_data_features.parquet' a été créé avec succès !")

# Test

In [ ]:

def get_recommandations(music_name, df, nombre_recommandations=5):
    try:
        chanson_reference = df.filter(col("name") == music_name).first()
        if chanson_reference is None:
            return f"Music '{music_name}' not found."

        cluster_id = chanson_reference["prediction"]
        print(f"Music '{music_name}' is in cluster '{cluster_id}'.")

        recommandations = df.filter(
            (col("prediction") == cluster_id) & (col("name") != music_name)
        ).select("name", "artists", "popularity")

        recommandations.show(truncate=True)

        return recommandations.orderBy(col("popularity").desc()).limit(nombre_recommandations)

    except Exception as e:
        return f"Error encountered : {e}"

test_music = "Shape of You" 
recos = get_recommandations(test_music, df_clustered)

if isinstance(recos, str):
    print(recos)
else:
    print(f"\nIf you like '{test_music}', maybe you'll like :")
    recos.show(truncate=True)